(course-alzheimer-conformational-engineering)=
# Module 27: Conformational Engineering

You can explore backbone conformations of the Alzheimer's-related **KLVFF**
peptide by imposing torsions. A beta-strand-like target is a geometric starting
point, not evidence of a stable bound state, fibril intercalation or an energy minimum.

> **🎯 Learning Outcomes**
> - Find explicit atom-index quartets for backbone torsions.
> - Set angular targets and apply a relative shift.
> - Preserve the input and check which atoms move.

In [1]:
import numpy as np
import molsysmt as msm
from molsysmt import pyunitwizard as puw

molsys = msm.build.build_peptide('KLVFF', engine='MolSysMT')
original_coordinates = puw.get_value(msm.get(molsys, coordinates=True), to_unit='nm').copy()
phi_quartets = np.asarray(msm.topology.get_dihedral_quartets(molsys, phi=True))
psi_quartets = np.asarray(msm.topology.get_dihedral_quartets(molsys, psi=True))

(course-alzheimer-conformational-engineering-reading)=
### 1. Reading Current Angles

Each quartet contains four atom **indices**, not group IDs. A terminal group can
lack phi or psi, so the number of torsions need not equal the number of groups.
The native builder supplies deterministic initial coordinates without force-field relaxation.

In [2]:
phi = msm.structure.get_dihedral_angles(molsys, dihedral_quartets=phi_quartets, pbc=False)
psi = msm.structure.get_dihedral_angles(molsys, dihedral_quartets=psi_quartets, pbc=False)
print('Phi (degrees):', puw.get_value(phi, to_unit='degrees'))
print('Psi (degrees):', puw.get_value(psi, to_unit='degrees'))

Phi (degrees): [[180. 180. 180. 180.]]
Psi (degrees): [[180. 180. 180. 180.]]


(course-alzheimer-conformational-engineering-target)=
### 2. Imposing a Backbone Shape

Assign the returned independent system. The default `in_place=False` preserves
`molsys`; ignoring the return would discard the edit. The editor derives the
moving covalent block on the fourth atom's side after cutting each central bond.
Here `pbc=False` describes one isolated peptide, without periodic reconstruction.

In [3]:
edited = msm.structure.set_dihedral_angles(
    molsys, dihedral_quartets=phi_quartets, angles='-135 degrees', pbc=False)
edited = msm.structure.set_dihedral_angles(
    edited, dihedral_quartets=psi_quartets, angles='135 degrees', pbc=False)
phi_after = msm.structure.get_dihedral_angles(edited, dihedral_quartets=phi_quartets, pbc=False)
psi_after = msm.structure.get_dihedral_angles(edited, dihedral_quartets=psi_quartets, pbc=False)
print('Target phi (degrees):', puw.get_value(phi_after, to_unit='degrees'))
print('Target psi (degrees):', puw.get_value(psi_after, to_unit='degrees'))

Target phi (degrees): [[-135. -135. -135. -135.]]
Target psi (degrees): [[135.         135.         135.         135.00000085]]


(course-alzheimer-conformational-engineering-shift)=
### 3. Shifting One Torsion

For the third amino acid (zero-based **group index 2**, VAL), choose the psi
quartet whose first atom is its N. Define the moving block explicitly and add
45 degrees. This operation does not resolve clashes or minimize energy.

In [4]:
nitrogen = msm.select(edited, selection='group_index==2 and atom_name=="N"')[0]
quartet = psi_quartets[psi_quartets[:, 0] == nitrogen][0]
components = msm.topology.get_covalent_blocks(
    edited, remove_bonds=quartet[1:3], output_type='sets')
moving = sorted(next(block for block in components if int(quartet[3]) in block))
before_shift = puw.get_value(msm.get(edited, coordinates=True), to_unit='nm').copy()
shifted = msm.structure.shift_dihedral_angles(
    edited, dihedral_quartets=quartet, shifts='45 degrees',
    blocks=[moving], pbc=False)
new_psi = msm.structure.get_dihedral_angles(shifted, dihedral_quartets=quartet, pbc=False)
print('Shifted VAL psi (degrees):', puw.get_value(new_psi, to_unit='degrees'))

Shifted VAL psi (degrees): [[180.]]


(course-alzheimer-conformational-engineering-challenge)=
### 🏆 Challenge: The Helix-Maker

Repeat the workflow with phi -57 degrees and psi -47 degrees; measure the
achieved angles and end-to-end C-alpha distance. Inspect the result with
{func}`molsysmt.basic.view`, following the [visualization tutorial](../../user/tools/basic/view.ipynb).
A visually plausible helix still needs clash and energetic assessment.

:::{key-takeaway}
:class: dropdown
Explicit quartets identify the measured torsion; explicit moving blocks identify
what an edit moves. A verified angle is a geometric result, not physical validation.
:::

(course-alzheimer-conformational-engineering-see-also)=
:::{seealso}
:class: dropdown
{func}`molsysmt.topology.get_dihedral_quartets`,
{func}`molsysmt.structure.set_dihedral_angles`,
{func}`molsysmt.structure.shift_dihedral_angles`,
and [Module 16: Structural Operations](../Common_Core/16_Structural_Operations.ipynb).
:::